# 00 — From data to a trustworthy baseline

## Start with the decision

**Question:** immediately before a scheduled call, which clients should be prioritized for
a term-deposit campaign?

This is a **prediction and ranking** problem: estimate who is likely to subscribe. It is
not a causal question about whether calling someone changes their outcome.

This notebook builds the foundation for trustworthy modeling: define the decision moment,
check the data contract, remove leakage, protect evaluation data, and create a simple baseline.


In [1]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# Known interoperability/UI warnings do not affect predictions or notebook execution.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (balanced_accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, log_loss, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # Return the course root when present, otherwise the notebook's directory.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # Set FAST_MODE=0 for full-size experiments; laptop mode is the default.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # The course ships with a local dataset; notebooks never access the network.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def prepare_precontact_frame(frame, drop_duration=True):
    '''Convert contact-level records into features valid immediately before the call.'''
    result = frame.copy()
    # UCI's campaign count includes the current contact, which has not happened at scoring time.
    result["prior_contacts_in_campaign"] = result.pop("campaign") - 1
    if (result["prior_contacts_in_campaign"] < 0).any():
        raise ValueError("campaign must be at least one in a contact-level record.")
    return result.drop(columns=LEAKAGE_COLUMNS, errors="ignore") if drop_duration else result

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and make the default frame safe for pre-call scoring.'''
    # Keep the raw contact-level frame only for audit demonstrations.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    return frame if include_duration else prepare_precontact_frame(frame)

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # Deterministic stratified 60/20/20 split; test stays sealed until notebook 07.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # Preprocessing is fitted only inside the enclosing training/CV pipeline.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    columns = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)
    return Pipeline([
        ("contract", BankMarketingInputValidator()),
        ("columns", columns),
    ])

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"log_loss": log_loss(y_true, probability),
            "brier_score": brier_score_loss(y_true, probability),
            "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["prior_contacts_in_campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
BRAND_COLOR = "#2a9d8f"
ACCENT_COLOR = "#e76f51"
sns.set_theme(style="whitegrid", context="notebook", palette=[BRAND_COLOR, ACCENT_COLOR, "#264653", "#f4a261"])
plt.rcParams.update({
    "axes.facecolor": "#fcfcfc",
    "figure.facecolor": "white",
    "axes.edgecolor": "#d9d9d9",
    "grid.color": "#e8e8e8",
    "grid.linewidth": 0.8,
    "axes.titleweight": "bold",
    "axes.labelweight": "medium",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})


{'FAST_MODE': True, 'CV_FOLDS': 3, 'seed': 42}


## 1. Load the contact-level data

Each row is one marketing contact and `y = 1` means the client subscribed. We use a local
snapshot so the results are reproducible. Loading data is not modeling yet: we must first
understand what each raw column means and when it exists.


In [2]:
path = bank_data_path()
raw = load_bank_data(include_duration=True)
print("cached path:", path)
print("shape:", raw.shape)
raw.head()


cached path: /Users/soroush/Desktop/Personal/Projects/Daneshkar/advanced_machine_learning/data/raw/bank-full.csv
shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0


## 2. State the data contract

A **data catalog** explains the dataset's purpose, source, grain, owner, and prediction
moment. A **data dictionary** explains individual columns. Together, they make the model's
assumptions visible before any training starts.


In [3]:
asset_catalog = pd.Series({
    "asset_name": "uci_bank_marketing",
    "local_path": str(path.relative_to(ROOT)),
    "source_url": "https://archive.ics.uci.edu/dataset/222/bank+marketing",
    "sha256": file_sha256(path),
    "rows": len(raw),
    "columns": raw.shape[1],
    "grain": "one row per marketing contact",
    "target": TARGET,
    "prediction_moment": "immediately before a scheduled call",
    "refresh_policy": "static course snapshot",
    "owner": "course maintainer",
    "steward": "simulated marketing-data steward",
}, name="value").to_frame()
display(asset_catalog)


,value
asset_name,uci_bank_marketing
local_path,data/raw/bank-full.csv
source_url,https://archive.ics.uci.edu/dataset/222/bank+m...
sha256,d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa...
rows,45211
columns,17
grain,one row per marketing contact
target,y
prediction_moment,immediately before a scheduled call
refresh_policy,static course snapshot


## 3. Decide which fields are safe

A feature is valid only if it exists at scoring time. `unknown` is a documented category,
not automatically missing data. `pdays = -1` means no previous contact. The table below
turns the raw schema into a feature contract by marking each field's role and availability.


In [4]:
field_roles = {
    "age": "client attribute", "job": "client attribute", "marital": "client attribute",
    "education": "client attribute", "default": "financial attribute",
    "balance": "financial attribute", "housing": "financial attribute", "loan": "financial attribute",
    "contact": "campaign context", "day": "campaign context", "month": "campaign context",
    "duration": "post-contact measurement", "campaign": "contact count including this call",
    "pdays": "campaign history", "previous": "campaign history", "poutcome": "campaign history",
    TARGET: "target",
}
schema = pd.DataFrame({
    "role": [field_roles[c] for c in raw.columns],
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(),
    "available_at_prediction": [c not in {"duration", "campaign", TARGET} for c in raw.columns],
    "example": [raw[c].iloc[0] for c in raw.columns],
})
display(schema)
print("target distribution:")
display(raw[TARGET].value_counts().rename("count").to_frame().assign(rate=lambda x: x["count"] / len(raw)))


,role,dtype,missing,unique,available_at_prediction,example
age,client attribute,int64,0,77,True,58
job,client attribute,object,0,12,True,management
marital,client attribute,object,0,3,True,married
education,client attribute,object,0,4,True,tertiary
default,financial attribute,object,0,2,True,no
balance,financial attribute,int64,0,7168,True,2143
housing,financial attribute,object,0,2,True,yes
loan,financial attribute,object,0,2,True,no
contact,campaign context,object,0,3,True,unknown
day,campaign context,int64,0,31,True,5


target distribution:


,count,rate
y,,
0,39922,0.883015
1,5289,0.116985


In [5]:
categorical = raw.select_dtypes(include="object").columns.tolist()
numeric = raw.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
print("categorical:", categorical)
print("numeric:", numeric)
display(raw[categorical].nunique().sort_values().rename("cardinality").to_frame())


categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
numeric: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


,cardinality
default,2
housing,2
loan,2
marital,3
contact,3
education,4
poutcome,4
job,12
month,12


## 4. Validate the input

A catalog is a promise; validation is evidence. Before modeling, verify the target, ranges,
sentinel values, and row count. If a critical rule fails in production, stop or quarantine
the batch rather than silently changing the rule.


In [6]:
quality_checks = pd.Series({
    "target is binary and non-null": raw[TARGET].notna().all() and set(raw[TARGET].unique()) <= {0, 1},
    "age is between 18 and 100": raw["age"].between(18, 100).all(),
    "day is between 1 and 31": raw["day"].between(1, 31).all(),
    "month uses documented abbreviations": set(raw["month"]) <= set("jan feb mar apr may jun jul aug sep oct nov dec".split()),
    "pdays is -1 or non-negative": ((raw["pdays"] == -1) | (raw["pdays"] >= 0)).all(),
    "campaign is at least one": raw["campaign"].ge(1).all(),
    "source row count is stable": len(raw) == 45_211,
}, name="passed").to_frame()
display(quality_checks)
assert quality_checks["passed"].all(), "Critical catalog quality contract failed"


,passed
target is binary and non-null,True
age is between 18 and 100,True
day is between 1 and 31,True
month uses documented abbreviations,True
pdays is -1 or non-negative,True
campaign is at least one,True
source row count is stable,True


### Use one contract for training and serving

The snapshot check protects this file. The pipeline validator protects every training,
cross-validation, and scoring batch. Keeping validation and preprocessing in one pipeline
prevents train/serve drift and prevents preprocessing leakage across CV folds.


In [7]:
from sklearn.base import BaseEstimator, TransformerMixin

DOCUMENTED_MONTHS = frozenset("jan feb mar apr may jun jul aug sep oct nov dec".split())

class BankMarketingInputValidator(BaseEstimator, TransformerMixin):
    """Fail fast when a training or scoring batch violates the feature contract."""
    required_columns = frozenset({"age", "day", "month", "pdays", "prior_contacts_in_campaign"})

    def _validate(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Bank Marketing preprocessing requires a pandas DataFrame.")
        missing = self.required_columns.difference(X.columns)
        if missing:
            raise ValueError(f"Missing required feature columns: {sorted(missing)}")
        if not X["age"].between(18, 100).all():
            raise ValueError("age must be between 18 and 100.")
        if not X["day"].between(1, 31).all():
            raise ValueError("day must be between 1 and 31.")
        if not set(X["month"].dropna()).issubset(DOCUMENTED_MONTHS):
            raise ValueError("month must use documented three-letter abbreviations.")
        if not ((X["pdays"] == -1) | (X["pdays"] >= 0)).all():
            raise ValueError("pdays must be -1 or non-negative.")
        if not X["prior_contacts_in_campaign"].ge(0).all():
            raise ValueError("prior_contacts_in_campaign must be non-negative.")
        if y is not None:
            target = pd.Series(y)
            if target.isna().any() or not set(target.unique()).issubset({0, 1}):
                raise ValueError("target must be binary and non-null.")

    def fit(self, X, y=None):
        self._validate(X, y)
        return self

    def transform(self, X):
        self._validate(X)
        return X

# This unfitted object is safe to place inside CV or a final model pipeline.
leakage_safe_preprocessor = make_preprocessor(load_bank_data().drop(columns=TARGET))
display(leakage_safe_preprocessor)


,steps,"[('contract', ...), ('columns', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 5. Audit leakage

Leakage is information unavailable when the prediction is made. `duration` is known during
or after the call, so it cannot be used before the call. The comparison below shows why a
large score improvement can be a warning sign rather than a deployment win.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

audit = prepare_precontact_frame(
    stratified_sample(raw, 12_000 if FAST_MODE else len(raw)), drop_duration=False
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

def leakage_audit(frame):
    X, y = frame.drop(columns=TARGET), frame[TARGET]
    model = Pipeline([
        ("preprocess", make_preprocessor(frame)),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ])
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["neg_log_loss", "balanced_accuracy"],
        n_jobs=-1,
    )
    return {
        "log_loss (lower is better)": -result["test_neg_log_loss"].mean(),
        "balanced_accuracy (higher is better)": result["test_balanced_accuracy"].mean(),
    }

leakage_comparison = pd.DataFrame({
    "with_duration (invalid)": leakage_audit(audit),
    "without_duration (deployable)": leakage_audit(audit.drop(columns="duration")),
}).T
leakage_comparison


,log_loss (lower is better),balanced_accuracy (higher is better)
with_duration (invalid),0.244025,0.664468
without_duration (deployable),0.304622,0.582498


### Keep only information available before the call

- Exclude `duration`.
- Replace raw `campaign` with `prior_contacts_in_campaign = campaign - 1`, because the raw
  count includes the current call.
- Use date, channel, and prior-history fields only when the serving workflow makes them known.

> Change the decision moment, and the feature contract must change too.


## 6. Protect the evaluation

- **Development:** fit models and run cross-validation.
- **Validation:** choose thresholds and compare finalists.
- **Test:** use once in Notebook 07 for the final estimate.

This course uses a stratified random split because there is no reliable timestamp or customer
identifier. In production, prefer temporal and customer-grouped splits when available.


In [9]:
clean = load_bank_data()  # duration excluded by default
development, validation, test = make_splits(clean, reduced=FAST_MODE)
split_summary = pd.DataFrame({
    "rows": [len(development), len(validation), len(test)],
    "positive_rate": [x[TARGET].mean() for x in (development, validation, test)],
}, index=["development", "validation", "test (sealed)"])
display(split_summary)
assert "duration" not in clean.columns
assert "campaign" not in clean.columns
assert clean["prior_contacts_in_campaign"].ge(0).all()
assert sum(len(x) for x in make_splits(clean, reduced=False)) == len(clean)


,rows,positive_rate
development,12000,0.117
validation,4000,0.117
test (sealed),4000,0.117


## 7. Establish a baseline

Logistic regression is our first benchmark: it is fast, interpretable, and uses the same
pipeline in every CV fold. A more complex model must improve a meaningful metric or business
decision—not just look sophisticated.


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

def split_xy(frame):
    return frame.drop(columns=TARGET), frame[TARGET]

baseline = Pipeline([
    ("preprocess", make_preprocessor(development)),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])

X_dev, y_dev = split_xy(development)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
dev_prob = cross_val_predict(baseline, X_dev, y_dev, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

baseline_summary = pd.Series({
    "log_loss": log_loss(y_dev, dev_prob),
    "brier_score": brier_score_loss(y_dev, dev_prob),
    "balanced_accuracy@0.5": balanced_accuracy_score(y_dev, dev_prob >= 0.5),
    "f1@0.5": f1_score(y_dev, dev_prob >= 0.5, zero_division=0),
}, name="development CV")
display(baseline_summary.to_frame())


,development CV
log_loss,0.307628
brier_score,0.088289
balanced_accuracy@0.5,0.562812
f1@0.5,0.224148


## 8. Evaluate probabilities and decisions

Accuracy hides failure on an imbalanced target. Use log loss and Brier score for probability
quality; use precision and recall to assess a chosen threshold. The best threshold depends
on contact capacity, conversion value, and the cost of missed opportunities.


In [11]:
dev_pred = (dev_prob >= 0.5).astype(int)
interpretation = pd.DataFrame({
    "predicted_positive_rate": [dev_pred.mean()],
    "actual_positive_rate": [y_dev.mean()],
    "precision": [precision_score(y_dev, dev_pred, zero_division=0)],
    "recall": [recall_score(y_dev, dev_pred, zero_division=0)],
}, index=["development CV"])
display(interpretation)

,predicted_positive_rate,actual_positive_rate,precision,recall
development CV,0.02725,0.117,0.593272,0.138177


### Read precision and recall together

- **Precision:** among selected clients, how many subscribe?
- **Recall:** among all subscribers, how many did we select?

A low predicted-positive rate means the threshold is selective. It does not by itself prove
poor calibration or excessive misses; inspect recall, probability metrics, and campaign capacity together.


## Key takeaway

Trustworthy ML begins before model tuning: define the decision moment, enforce feature
availability, keep preprocessing inside CV, and preserve a sealed test set.
